# CIT Fraud Detection Pipeline
Step-by-step execution with column names printed after every sub-step.

**Run this notebook from inside the `cit/` folder.**

> **Note:** `standardize_columns`, `enrich_taxpayer_names`, and `validate_and_clean_cit_data`
> are currently commented out in the pipeline and are not available for import.
> Those steps are skipped below. The active pipeline flow is:
> `load → validate_required_columns → create_aggregated_columns → reorganize → rule_flags → xgboost → justification`

In [1]:
import os, sys, time, warnings, uuid
import pandas as pd
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

# ── SET working directory to cit/ folder ───────────────────────────────────
CIT_DIR = os.path.dirname(os.path.abspath('cit_fraud_pipeline_with_timer.py'))
os.chdir(CIT_DIR)
sys.path.insert(0, CIT_DIR)
sys.path.insert(0, os.path.dirname(CIT_DIR))   # project root

print(f'Working directory: {os.getcwd()}')

Working directory: c:\Users\sudip\OneDrive\Desktop\project_IRC\cit


In [2]:
# Import only functions that are active (not commented out) in the pipeline
from cit_fraud_pipeline_with_timer import (
    get_output_dir,
    load_and_preprocess_data,
    # standardize_columns       ← commented out in pipeline
    validate_required_columns,
    create_aggregated_columns,
    # enrich_taxpayer_names     ← commented out in pipeline
    reorganize_columns,
    # validate_and_clean_cit_data  ← commented out in pipeline
    apply_cit_flag_rules,
    process_and_predict_fraud_xgboost,
    process_cit_fraud_data,
    FraudDetectionSystem,
)

OUTPUT_DIR = get_output_dir()
print(f'Output directory: {OUTPUT_DIR}')

Output directory: c:\Users\sudip\OneDrive\Desktop\project_IRC\cit\final_output


---
# SCRIPT 1 — Data Preprocessing
Active sub-steps: **Load → Validate Required Columns → Create Aggregated Columns → Reorganize → Save**

> Sub-steps 1.2 (standardize_columns) and 1.5 (enrich_taxpayer_names) are disabled in
> the pipeline — the raw CIT data is expected to already have clean, standardised column names.

### Sub-step 1.1 — Load & Preprocess
Reads raw file from `data/`, drops blank/unnamed columns.

In [3]:
cit = load_and_preprocess_data()

print(f'\n=== COLUMNS AFTER SUB-STEP 1.1 — Load & Preprocess ({len(cit.columns)}) ===')
print(cit.columns.tolist())

--- Step 1: Loading and preprocessing data ---
Multiple files found:
  - 25.05.21.05 TIN Registrations.csv
  - cit_all.csv
  - gst.parquet
  - swt.parquet
Auto-selected most recent CIT file: cit_all.csv
Loaded: cit_all.csv | shape: (70575, 238)
After dropping empty columns: (70575, 237)

=== COLUMNS AFTER SUB-STEP 1.1 — Load & Preprocess (237) ===
['tin', 'taxpayer', 'tax_account_no', 'tax_type', 'tax_period_year', 'assessment_no', 'received_date', 'entry_date', 'due_date', 'form_no', 'form_version_no', 'irc_form_version_no', 'form_description', 'gross_sales_cash_or_credit', 'gross_contract_and_sub_con', 'partnership_distribution_i', 'distributions_from_trusts', 'oil_pipeline_tariffs_and_r', 'dividend_income', 'exchange_gains_or_losses', 'interest_income', 'rental_income', 'royalty_income', 'other_gross_income', 'total_gross_income', 'cost_of_goods_sold', 'rented_property_expenses_i', 'resource_operations_joint', 'amortisation', 'advertising_and_promotion', 'bad_debts_written_off', 'bo

### Sub-step 1.2 — Standardize Column Names *(SKIPPED — disabled in pipeline)*
This function is currently commented out. The raw data is expected to have pre-standardised names.

In [4]:
#standardize_columns is commented out in cit_fraud_pipeline_with_timer.py
#cit = standardize_columns(cit)
print('Sub-step 1.2 SKIPPED — standardize_columns is disabled in the pipeline.')
print(f'Column count unchanged: {len(cit.columns)}')

Sub-step 1.2 SKIPPED — standardize_columns is disabled in the pipeline.
Column count unchanged: 237


### Sub-step 1.3 — Validate Required Columns
Checks all 29 required columns are present. Stops pipeline if any are missing.

In [5]:
is_valid, message = validate_required_columns(cit)
print(message)
if not is_valid:
    raise SystemExit('Pipeline stopped: missing required columns — see message above.')

print(f'\n=== COLUMNS AFTER SUB-STEP 1.3 — Validate Required (no change) ({len(cit.columns)}) ===')
print(cit.columns.tolist())


--- Step 3: Validating required columns ---
Checking required columns...


Validating columns: 100%|██████████| 29/29 [00:00<?, ?col/s]

All required columns are present.

=== COLUMNS AFTER SUB-STEP 1.3 — Validate Required (no change) (237) ===
['tin', 'taxpayer', 'tax_account_no', 'tax_type', 'tax_period_year', 'assessment_no', 'received_date', 'entry_date', 'due_date', 'form_no', 'form_version_no', 'irc_form_version_no', 'form_description', 'gross_sales_cash_or_credit', 'gross_contract_and_sub_con', 'partnership_distribution_i', 'distributions_from_trusts', 'oil_pipeline_tariffs_and_r', 'dividend_income', 'exchange_gains_or_losses', 'interest_income', 'rental_income', 'royalty_income', 'other_gross_income', 'total_gross_income', 'cost_of_goods_sold', 'rented_property_expenses_i', 'resource_operations_joint', 'amortisation', 'advertising_and_promotion', 'bad_debts_written_off', 'borrowing_expenses', 'commissions', 'contract_employees', 'consultancy_fees', 'consumables', 'depreciation', 'development_levy', 'directors_fees_and_expens', 'entertainment_expenses', 'foreign_exchange_losses_or', 'gifts_and_donations', 'insura

### Sub-step 1.4 — Create Aggregated Columns
Adds derived/aggregated financial columns (totals, sums, etc.).

In [6]:
cit = create_aggregated_columns(cit)

print(f'\n=== COLUMNS AFTER SUB-STEP 1.4 — Aggregated Columns ({len(cit.columns)}) ===')
print(cit.columns.tolist())

--- Creating aggregated columns ---
Checking and creating aggregated columns...
Created column: total_sales_revenue
Created column: total_distributions_royalties
Created column: total_investment_income
Created column: total_other_income
Created column: non_operating_income
Created column: total_cost_of_goods_sold
Created column: total_property_rental_expenses
Created column: total_resource_operations
Created column: total_depreciation_amortization
Created column: total_marketing_promotion
Created column: total_financial_expenses
Created column: total_employee_expenses
Created column: total_professional_fees
Created column: total_operational_expenses
Created column: total_amortization_depreciation
Created column: total_non_allowable_capital_expenses
Created column: total_provisions_taxes
Created column: total_non_allowable_donations_legal
Created column: total_goodwill_formation_expenses
Created column: total_recouped_lease_premiums
Created column: total_excess_fees_interest
Created col

### Sub-step 1.5 — Enrich Taxpayer Names *(SKIPPED — disabled in pipeline)*
This function is currently commented out.

In [7]:
# enrich_taxpayer_names is commented out in cit_fraud_pipeline_with_timer.py
# cit = enrich_taxpayer_names(cit)
print('Sub-step 1.5 SKIPPED — enrich_taxpayer_names is disabled in the pipeline.')
print(f'Column count unchanged: {len(cit.columns)}')

Sub-step 1.5 SKIPPED — enrich_taxpayer_names is disabled in the pipeline.
Column count unchanged: 283


### Sub-step 1.6 — Reorganize Columns
Reorders columns: TIN and taxpayer name first, then financials.

In [8]:
cit = reorganize_columns(cit)

print(f'\n=== COLUMNS AFTER SUB-STEP 1.6 — Reorganize ({len(cit.columns)}) ===')
print(cit.columns.tolist())


--- Step 5: Reorganizing columns ---


Reorganizing: 100%|██████████| 283/283 [00:00<?, ?col/s]


=== COLUMNS AFTER SUB-STEP 1.6 — Reorganize (283) ===
['tin', 'taxpayer', 'tax_account_no', 'tax_type', 'tax_period_year', 'assessment_no', 'received_date', 'entry_date', 'due_date', 'form_no', 'form_version_no', 'irc_form_version_no', 'form_description', 'gross_sales_cash_or_credit', 'gross_contract_and_sub_con', 'partnership_distribution_i', 'distributions_from_trusts', 'oil_pipeline_tariffs_and_r', 'dividend_income', 'exchange_gains_or_losses', 'interest_income', 'rental_income', 'royalty_income', 'other_gross_income', 'total_gross_income', 'cost_of_goods_sold', 'rented_property_expenses_i', 'resource_operations_joint', 'amortisation', 'advertising_and_promotion', 'bad_debts_written_off', 'borrowing_expenses', 'commissions', 'contract_employees', 'consultancy_fees', 'consumables', 'depreciation', 'development_levy', 'directors_fees_and_expens', 'entertainment_expenses', 'foreign_exchange_losses_or', 'gifts_and_donations', 'insurance', 'interest_expense_png', 'interest_expense_forei

### Sub-step 1.7 — Save Script 1 Output

In [9]:
cit.to_parquet(os.path.join(OUTPUT_DIR, 'cit_preprocessed_data.parquet'), index=False)
cit.to_csv(os.path.join(OUTPUT_DIR, 'cit_preprocessed_data.csv'), index=False)
print(f'Saved: cit_preprocessed_data.parquet + .csv  |  shape: {cit.shape}')

Saved: cit_preprocessed_data.parquet + .csv  |  shape: (70575, 283)


---
# SCRIPT 2 — Data Validation & Cleaning *(SKIPPED — disabled in pipeline)*
`validate_and_clean_cit_data` is currently commented out in the pipeline.
The preprocessed data is passed directly to Script 3.

In [ ]:
# validate_and_clean_cit_data is commented out in cit_fraud_pipeline_with_timer.py
# cleaned_data = validate_and_clean_cit_data(cit)

# Use preprocessed data directly as cleaned_data for downstream steps
cleaned_data = cit.copy()
print('Script 2 SKIPPED — validate_and_clean_cit_data is disabled in the pipeline.')
print(f'Passing preprocessed data forward | shape: {cleaned_data.shape}')
print(f'\n=== COLUMNS (same as Script 1 output) ({len(cleaned_data.columns)}) ===')
print(cleaned_data.columns.tolist())

---
# SCRIPT 3A — Apply CIT Flag Rules
Reads `cit_preprocessed_data.csv`, adds `rule_1_valid` through `rule_21_valid` + `sum_of_rules`.
Saves → `final_output/cit_with_rule_violations.csv`

In [ ]:
# apply_cit_flag_rules reads cit_preprocessed_data.csv from OUTPUT_DIR
df_rules = apply_cit_flag_rules()

print(f'\n=== COLUMNS AFTER SCRIPT 3A — Rule Flags ({len(df_rules.columns)}) ===')
print(df_rules.columns.tolist())

---
# SCRIPT 3B — XGBoost Fraud Prediction
Loads XGBoost model + scaler + feature columns, scales features, adds `predicted_fraud`.
Saves → `final_output/cit_final_fraud_prediction.csv`

In [ ]:
df_prediction = process_and_predict_fraud_xgboost()

print(f'\n=== COLUMNS AFTER SCRIPT 3B — XGBoost Prediction ({len(df_prediction.columns)}) ===')
print(df_prediction.columns.tolist())

---
# SCRIPT 4 — Fraud Justification
Reads `cit_final_fraud_prediction.csv`, adds `Justification` column.
Saves to MySQL (`cit_fraud_justification` table). Falls back to CSV if DB is unavailable.

In [ ]:
upload_batch_id = str(uuid.uuid4())
uploaded_at     = datetime.now()
print(f'Batch ID : {upload_batch_id}')
print(f'Timestamp: {uploaded_at}')

db_saved, record_count = process_cit_fraud_data(
    upload_batch_id=upload_batch_id,
    uploaded_at=uploaded_at
)
print(f'\nDB saved: {db_saved}  |  Records processed: {record_count}')

In [ ]:
# Read back the justification output to inspect columns
justification_parq = os.path.join(OUTPUT_DIR, 'cit_fraud_justification.parquet')
justification_csv  = os.path.join(OUTPUT_DIR, 'cit_fraud_with_justification.csv')

if os.path.exists(justification_parq):
    df_step4 = pd.read_parquet(justification_parq)
    print('Reading from: cit_fraud_justification.parquet')
elif os.path.exists(justification_csv):
    df_step4 = pd.read_csv(justification_csv)
    print('Reading from: cit_fraud_with_justification.csv (fallback — DB write failed)')
else:
    df_step4 = None
    print('No local justification file found — data saved to MySQL only.')

if df_step4 is not None:
    print(f'\n=== COLUMNS AFTER SCRIPT 4 — Justification ({len(df_step4.columns)}) ===')
    print(df_step4.columns.tolist())
    print(f'\nShape: {df_step4.shape}')
    fraud_count = (df_step4['predicted_fraud'].str.lower() == 'fraud').sum()
    print(f'Fraud   : {fraud_count:,}')
    print(f'Non-Fraud: {len(df_step4) - fraud_count:,}')

---
## Full Column Summary

In [ ]:
steps = [
    ('After Sub-step 1.1  Load & Preprocess        ', cit),
    ('After Sub-step 1.3  Validate (no change)     ', cit),
    ('After Sub-step 1.4  Aggregated Columns       ', cit),
    ('After Sub-step 1.6  Reorganize               ', cit),
    ('After Script 2      Validation (skipped)     ', cleaned_data),
    ('After Script 3A     Rule Flags               ', df_rules),
    ('After Script 3B     XGBoost Prediction       ', df_prediction),
]

print('=== COLUMN COUNT SUMMARY ===')
for label, df in steps:
    print(f'  {label}: {len(df.columns)} columns')

print('\n=== NEW COLUMNS ADDED BY RULE FLAGS (Script 3A) ===')
rule_cols = [c for c in df_rules.columns if c not in cleaned_data.columns]
print(rule_cols)

print('\n=== NEW COLUMNS ADDED BY XGBOOST (Script 3B) ===')
pred_cols = [c for c in df_prediction.columns if c not in df_rules.columns]
print(pred_cols)

if df_step4 is not None:
    print('\n=== NEW COLUMNS ADDED BY JUSTIFICATION (Script 4) ===')
    just_cols = [c for c in df_step4.columns if c not in df_prediction.columns]
    print(just_cols)